# [Semantic Kernel 05 - Multiple Agents](https://devblogs.microsoft.com/semantic-kernel/introducing-agents-in-semantic-kernel/)
Up until today, we've demonstrated how you could use components of Semantic Kernel to build agents. With just a few lines of code, you can use a chat completion model to answer user’s questions and to automatically invoke plugins as necessary.<br/>
What was missing, however, was a first-class agent abstraction. Not only would this simplify code by consolidating logic, but it would also ensure that there is a common contract on how to interact with an agent. This may seem like a small feat, but it has allowed us to build a multi-agent framework that allows agents to coordinate with one another while also simplifying the code you need to write.<br/>
With our latest Python (1.6.0) and .NET releases (1.18.0 RC1), Semantic Kernel now provides a first-class abstraction for agents. This reduces much of the complexity required to build a standard chat experience while also providing a standardized API to interact with them. With this release, we’re providing two out-of-the-box agents: Assistant API agents and Chat completion agents.

# Constants and Libraries

In [13]:
import os, json, sys, random
from dotenv import load_dotenv # requires python-dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior # Auto(), Required() or NoneInvoke()
from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.agents.strategies.termination.termination_strategy import TerminationStrategy
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.contents.chat_message_content import ChatMessageContent

from IPython.display import Markdown, display # nothing to pip install
sys.path.append('./common')
from utils import *  

load_dotenv("./../config/credentials_my.env")

data_path = "./data/data.json"

report_agent_name      = "Report_Agent"
researcher_agent_name  = "Researcher_Agent"
ethicist_agent_name    = "Ethicist_Agent"
economist_agent_name   = "Economist_Agent"
policymaker_agent_name = "PolicyMaker_Agent"

report_service_id      = "Report_Agent"
researcher_service_id  = "Researcher_Agent"
ethicist_service_id    = "Ethicist_Agent"
economist_service_id   = "Economist_Agent"
policymaker_service_id = "PolicyMaker_Agent"

report_system_message      = "You are responsible for creating a report by extracting insights from the chat history."
researcher_system_message  = "You explore and describe the potential capabilities and advancements of the topic."
ethicist_system_message    = "You evaluate the ethical implications of the topic, based on the research findings."
economist_system_message   = "You analyze the economic impact of the topic, based on the ethical evaluations."
policymaker_system_message = "You develop policies to manage the topic, based on the economic analysis."


chatcompletion_service_id   = "chatcompletion_service_id"
instructions                = "you are a clever agent"
content                     = "Toggle the status of my second light."

print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Data

In [2]:
# Open the JSON file and load its content into the 'topics' variable  
with open(data_path, 'r') as file:  
    topics = json.load(file)
    
topic = topic_to_string(random.choice(topics))
print(topic)

Cambiamento climatico e sostenibilitÃ  ambientale:
- Riduzione delle emissioni di gas serra
- Energie rinnovabili e transizione energetica
- Conservazione della biodiversitÃ 
- Inquinamento e gestione dei rifiuti


# Define the Kernel

In [3]:
kernel = Kernel()
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x00000288C8085D30>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Create an AzureChatCompletion AI Service and add it to the Kernel

In [4]:
kernel.add_service(AzureChatCompletion(service_id=chatcompletion_service_id))
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='chatcompletion_service_id', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x00000288C74E8A40>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x00000288C8085D30>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Enable planning with Function Calling set as Auto()

In [5]:
execution_settings = AzureChatPromptExecutionSettings()
execution_settings.function_choice_behavior= FunctionChoiceBehavior.Auto(auto_invoke=True) # Auto(), Required() or NoneInvoke()
execution_settings

AzureChatPromptExecutionSettings(service_id=None, extension_data={}, function_choice_behavior=FunctionChoiceBehavior(enable_kernel_functions=True, maximum_auto_invoke_attempts=5, filters=None, type_=<FunctionChoiceType.AUTO: 'auto'>), ai_model_id=None, frequency_penalty=None, logit_bias=None, max_tokens=None, number_of_responses=None, presence_penalty=None, seed=None, stop=None, stream=False, temperature=None, top_p=None, user=None, store=None, metadata=None, response_format=None, function_call=None, functions=None, messages=None, function_call_behavior=None, parallel_tool_calls=True, tools=None, tool_choice=None, structured_json_response=False, stream_options=None, extra_body=None)

# Set the logging level for  semantic_kernel.kernel to DEBUG
One of the main benefits of using Semantic Kernel is that it supports enterprise-grade services.<br/>
In this sample, we add the logging service to the kernel to help debug the AI agent.

In [6]:
import logging

logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
    
logging.getLogger("kernel").setLevel(logging.DEBUG)

# Create the agents
The agent `service_id` specified in `ChatCompletionAgent` must match one of the services defined in `Kernel.services`

In [7]:
user_proxy = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="user_proxy",
    description="Orchestrates the multiple personas involved in a discussion",
    kernel=kernel,    
    instructions="""
        Tu sei incaricato di iniziare e moderare la conversazione. 
        Il tuo ruolo è di fornire una piattaforma dove le persone possono interagire, mostrando le loro inclinazioni e abilità.
        Tuo incarico: facilita una conversazione discorsiva e coinvolgente fra Mauro, Aleksa, Gabriel e Federica senza imporre alcun pregiudizio.
    """
)

In [8]:
mauro = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="mauro",
    description="A Microsoft Cloud Solution Architect expert at Data Engineering",
    kernel=kernel,    
    instructions="""
        Tu sei Mauro. Mauro è un Cloud Solution Architect specializzato in Data Engineering, 
        esperto nella progettazione e implementazione di soluzioni per la gestione e l'analisi di grandi volumi di dati, 
        con un forte impegno verso l'ottimizzazione delle prestazioni e l'integrità dei dati.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """
)

In [9]:
aleksa = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="aleksa",
    description="A Microsoft Cloud Solution Architect expert at Data Science and Generative AI",
    kernel=kernel,    
    instructions="""
        Tu sei Aleksa. Sei un Cloud Solution Architect specializzata in Data Science, con una profonda conoscenza 
        delle tecniche di machine learning, intelligenza artificiale e generative AI, appassionata di trasformare dati complessi 
        in insights utili per guidare le decisioni aziendali.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """
)

In [10]:
gabriel = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="gabriel",
    description="A Microsoft Cloud Solution Architect expert at Security",
    kernel=kernel,    
    instructions="""
        Tu sei Gabriel. Sei Cloud Solution Architect esperto di sicurezza, dedicato a progettare e implementare 
        soluzioni di sicurezza cloud robuste e scalabili, con un forte impegno verso la protezione dei dati e la conformità alle normative.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """
)

In [11]:
federica = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="federica",
    description="A Microsoft Cloud Solution Architect expert in Development Tools",
    kernel=kernel,    
    instructions="""
        Tu sei Federica. Sei Cloud Solution Architect specializzata in tecnologie di sviluppo, con una vasta esperienza 
        nella creazione di applicazioni cloud-native e microservizi, sempre alla ricerca di innovazioni che migliorino 
        l'efficienza e la scalabilità delle soluzioni software.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """
)

# Termination Strategy definition

In [14]:
class ApprovalTerminationStrategy(TerminationStrategy):
    """A strategy for determining when an agent should terminate."""

    async def should_agent_terminate(self, agent, history):
        """Check if the agent should terminate."""
        return "approved" in history[-1].content.lower()

# Create Group Chat

In [15]:
chat = AgentGroupChat(
    agents = [user_proxy, mauro, aleksa, gabriel, federica],
    termination_strategy=ApprovalTerminationStrategy(agents=[user_proxy], maximum_iterations=10),
)

# Create a user message and add it to a blank history

In [16]:
input = "Your objective is to create the agenda for a 2 hours presentation on the most recent Azure technologies."

await chat.add_chat_message(ChatMessageContent(role=AuthorRole.USER, content=input))
print(f"# {AuthorRole.USER}: '{input}'")

# AuthorRole.USER: 'Your objective is to create the agenda for a 2 hours presentation on the most recent Azure technologies.'


In [17]:
async for content in chat.invoke():
    print(f"# {content.role} - {content.name or '*'}: '{content.content}'")

# AuthorRole.ASSISTANT - user_proxy: 'Sure, I can help with that! Let's gather everyone and start brainstorming ideas for the agenda.

Hey Mauro, Aleksa, Gabriel, and Federica, thank you for joining the conversation. We need to create an agenda for a 2-hour presentation on the most recent Azure technologies. I'll start by suggesting a structure, and then we can discuss and refine it based on your thoughts and expertise.

**Proposed Agenda:**

1. **Introduction (10 minutes)**
   - Welcome and Objectives
   - Overview of Azure platform

2. **Azure AI and Machine Learning (20 minutes)**
   - Latest updates and features
   - Use cases and industry applications

3. **Azure DevOps and GitHub Integration (20 minutes)**
   - Enhancements in DevOps tools 
   - Workflow improvements and integration insights

4. **Azure Security and Compliance (20 minutes)**
   - New security features
   - Best practices for compliance

5. **Break (10 minutes)**

6. **Azure Kubernetes Service (AKS) (20 minutes)**

In [18]:
print(f"# IS COMPLETE: {chat.is_complete}")

# IS COMPLETE: False


# Generate the agent response(s)

In [ ]:
async for response in agent.invoke(history):
  print(response)

# Additional tests. Run multiple times to toggle the first light.

In [ ]:
history = ChatHistory() # initially blank
history.add_user_message("Toggle the first light and give me the status of all my lights.")

async for response in agent.invoke(history):
  print(response)

In [ ]:
history

In [ ]:
for cmc in history.messages: # ChatMessageContent
    if not cmc.inner_content is None:
        for choice in cmc.inner_content.choices:
            for tc in choice.message.tool_calls:
                print (f"Call {tc.function.name}({tc.function.arguments})")